# 原始G17两份NAICS3344版式异常

只读封存来源Recorder中的2012-03和2016-03两份发布文件，不解析因子数值、市场结果或模型。

In [1]:
from pathlib import Path
import hashlib,json,os
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace missing")
    root=root.parent
card=root/"research/semiconductor_ip_g17_layout_exception_20261022"
plan=json.loads((card/"plan.json").read_text());source=root/plan['source_card']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(source/"summary.json")==plan['source_summary_sha256']
assert sha(source/"verification_manifest.json")==plan['source_verification_manifest_sha256']
assert sha(root/"research/semiconductor_ip_original_g17_bridge_20261022/source_row_audit.json")==plan['prior_audit_result_sha256']
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
qlib.init(provider_uri=json.loads((source/"summary.json").read_text())['provider_uri'],region='us',kernels=1)
objects,audit=load_verified_recorder_objects(source,expected_recorder_id=plan['source_recorder_id'],artifacts=[plan['source_raw_artifact']],expected_manifest_sha256=plan['source_verification_manifest_sha256'],expected_artifact_sha256={plan['source_raw_artifact']:plan['source_raw_artifact_sha256']})
raw=objects[plan['source_raw_artifact']]
rows=[]
for name in plan['target_files']:
    body=raw[name];text=body.decode('utf-8','replace');lines=text.splitlines();hits=[]
    for i,line in enumerate(lines):
        if 'semiconductor' not in line.lower() and '3344' not in line:continue
        if len(hits)>=18:break
        before='\n'.join(lines[:i]);table=max(before.rfind('Table 1'),before.rfind('Table 2'))
        hits.append({'line_number':i+1,'nearest_table_header':before[table:table+45] if table>=0 else None,
                     'context':'\n'.join(lines[max(0,i-1):min(len(lines),i+3)])[:600]})
    rows.append({'file':name,'source_sha256':hashlib.sha256(body).hexdigest(),'hit_count_capped18':len(hits),'hits':hits})
result={'status':'two_source_layouts_inspected_no_numeric_feature_values','source_recorder_id':plan['source_recorder_id'],'plan_sha256':sha(card/'plan.json'),'documents':rows,'new_source_GETs':0,'market_outcome_reads':0,'numeric_factor_values':0,'model_fits':0,'native_accounts':0}
(card/'layout_scan.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps({'status':result['status'],'hits':{x['file']:x['hit_count_capped18'] for x in rows}}))

{"status": "two_source_layouts_inspected_no_numeric_feature_values", "hits": {"20120316_sup.txt": 7, "20160316_sup.txt": 7}}


## 中文结论

2012-03-16与2016-03-16的原始`Table 1D`均含NAICS3344工业生产行；续行前多一空行是前一卡漏报的版式原因。见`layout_scan.json`、`adjudication.json`和`conclusion.md`。合计192/192有行，尚无数值/基年/首收资格。